# Previsão de Volume de Atendimentos — 2026
### Modelos: ARIMA · Prophet · Random Forest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

df_raw = pd.read_csv('../Bases/VolumeDeAtendimentos.csv', sep=';', parse_dates=['data'])
df_raw.head()

In [ ]:
# Agrega volume total diário (todas as empresas e produtos)
serie = (
    df_raw
    .groupby('data')['volume_atendimentos']
    .sum()
    .asfreq('D')
    .rename('volume')
)
print(f"Período: {serie.index.min().date()} → {serie.index.max().date()}")
print(f"Total de dias: {len(serie)}")
print(f"Valores nulos: {serie.isna().sum()}")
serie.plot(title='Volume Diário de Atendimentos (histórico)', ylabel='Volume');
plt.tight_layout()
plt.show()

## 1. ARIMA

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# Teste de estacionariedade
result = adfuller(serie.dropna())
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print("Série estacionária:", result[1] < 0.05)

In [ ]:
from pmdarima import auto_arima

# auto_arima busca os melhores parâmetros (p,d,q)(P,D,Q,s=7)
print("Buscando melhores parâmetros SARIMA (pode levar alguns minutos)...")
arima_model = auto_arima(
    serie,
    seasonal=True,
    m=7,                  # sazonalidade semanal
    stepwise=True,
    information_criterion='aic',
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    max_p=3, max_q=3,
    max_P=2, max_Q=2,
    d=None, D=None
)
print("\nMelhor modelo:", arima_model.order, arima_model.seasonal_order)

In [ ]:
# Previsão para todos os dias de 2026
future_index_arima = pd.date_range('2026-01-01', '2026-12-31', freq='D')
n_periods = len(future_index_arima)

forecast_arima, conf_int_arima = arima_model.predict(
    n_periods=n_periods,
    return_conf_int=True,
    alpha=0.05
)

df_arima = pd.DataFrame({
    'data': future_index_arima,
    'previsao_arima': forecast_arima,
    'lower_arima': conf_int_arima[:, 0],
    'upper_arima': conf_int_arima[:, 1]
}).set_index('data')

df_arima.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
serie.iloc[-180:].plot(ax=ax, label='Histórico (6m)', color='steelblue')
df_arima['previsao_arima'].plot(ax=ax, label='Previsão ARIMA 2026', color='tomato')
ax.fill_between(df_arima.index,
                df_arima['lower_arima'],
                df_arima['upper_arima'],
                alpha=0.25, color='tomato', label='IC 95%')
ax.set_title('Previsão ARIMA — Volume Diário 2026')
ax.set_ylabel('Volume de Atendimentos')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
plt.tight_layout()
plt.show()

print(f"\nTotal previsto 2026 (ARIMA): {df_arima['previsao_arima'].sum():,.0f}")

## 2. Prophet

In [ ]:
from prophet import Prophet

# Prophet exige colunas 'ds' e 'y'
df_prophet_train = serie.reset_index().rename(columns={'data': 'ds', 'volume': 'y'})

model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    interval_width=0.95,
    changepoint_prior_scale=0.1
)

# Feriados brasileiros relevantes
from prophet.make_holidays import make_holidays_df
br_holidays = make_holidays_df(year_list=[2021, 2022, 2023, 2024, 2025, 2026], country='BR')
model_prophet.add_country_holidays(country_name='BR')

model_prophet.fit(df_prophet_train)
print("Prophet treinado com sucesso.")

In [ ]:
future_prophet = model_prophet.make_future_dataframe(periods=365, freq='D')
forecast_prophet = model_prophet.predict(future_prophet)

df_prophet = (
    forecast_prophet[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    .rename(columns={'ds': 'data', 'yhat': 'previsao_prophet',
                     'yhat_lower': 'lower_prophet', 'yhat_upper': 'upper_prophet'})
    .set_index('data')
)
df_prophet_2026 = df_prophet['2026-01-01':'2026-12-31']
df_prophet_2026.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
serie.iloc[-180:].plot(ax=ax, label='Histórico (6m)', color='steelblue')
df_prophet_2026['previsao_prophet'].plot(ax=ax, label='Previsão Prophet 2026', color='darkorange')
ax.fill_between(df_prophet_2026.index,
                df_prophet_2026['lower_prophet'],
                df_prophet_2026['upper_prophet'],
                alpha=0.25, color='darkorange', label='IC 95%')
ax.set_title('Previsão Prophet — Volume Diário 2026')
ax.set_ylabel('Volume de Atendimentos')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
plt.tight_layout()
plt.show()

print(f"\nTotal previsto 2026 (Prophet): {df_prophet_2026['previsao_prophet'].sum():,.0f}")

In [ ]:
# Componentes da decomposição do Prophet
fig_comp = model_prophet.plot_components(forecast_prophet)
plt.tight_layout()
plt.show()

## 3. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

def build_features(idx: pd.DatetimeIndex) -> pd.DataFrame:
    """Engenharia de features temporais para o Random Forest."""
    df_feat = pd.DataFrame(index=idx)
    df_feat['day_of_year']  = idx.dayofyear
    df_feat['day_of_week']  = idx.dayofweek      # 0=seg, 6=dom
    df_feat['week_of_year'] = idx.isocalendar().week.astype(int)
    df_feat['month']        = idx.month
    df_feat['quarter']      = idx.quarter
    df_feat['year']         = idx.year
    df_feat['is_weekend']   = (idx.dayofweek >= 5).astype(int)
    # Seno/cosseno para capturar ciclicidade
    df_feat['sin_dow'] = np.sin(2 * np.pi * df_feat['day_of_week'] / 7)
    df_feat['cos_dow'] = np.cos(2 * np.pi * df_feat['day_of_week'] / 7)
    df_feat['sin_doy'] = np.sin(2 * np.pi * df_feat['day_of_year'] / 365.25)
    df_feat['cos_doy'] = np.cos(2 * np.pi * df_feat['day_of_year'] / 365.25)
    return df_feat

# Features de lags e médias móveis sobre o histórico
df_rf = serie.to_frame()
for lag in [1, 2, 3, 7, 14, 21, 28, 30]:
    df_rf[f'lag_{lag}'] = df_rf['volume'].shift(lag)

for window in [7, 14, 30]:
    df_rf[f'roll_mean_{window}'] = df_rf['volume'].shift(1).rolling(window).mean()
    df_rf[f'roll_std_{window}']  = df_rf['volume'].shift(1).rolling(window).std()

feat_temporais = build_features(df_rf.index)
df_rf = pd.concat([df_rf, feat_temporais], axis=1).dropna()

X = df_rf.drop(columns=['volume'])
y = df_rf['volume']

# Treino: 2021–2024 | Validação: 2025
X_train = X[X.index.year < 2025]
y_train = y[y.index.year < 2025]
X_val   = X[X.index.year == 2025]
y_val   = y[y.index.year == 2025]

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred_val = rf_model.predict(X_val)
mae  = mean_absolute_error(y_val, y_pred_val)
rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
print(f"Validação 2025 → MAE: {mae:,.1f} | RMSE: {rmse:,.1f}")

In [ ]:
# Previsão iterativa para 2026 (usa previsões anteriores como lags)
future_index_rf = pd.date_range('2026-01-01', '2026-12-31', freq='D')

# Histórico completo para usar como base dos lags
historico = serie.copy()

preds_rf = []
for data in future_index_rf:
    feat_row = build_features(pd.DatetimeIndex([data]))

    # Lags calculados a partir do histórico acumulado
    for lag in [1, 2, 3, 7, 14, 21, 28, 30]:
        ref_date = data - pd.Timedelta(days=lag)
        feat_row[f'lag_{lag}'] = historico.get(ref_date, np.nan)

    for window in [7, 14, 30]:
        past_vals = historico[historico.index < data].iloc[-window:]
        feat_row[f'roll_mean_{window}'] = past_vals.mean() if len(past_vals) == window else np.nan
        feat_row[f'roll_std_{window}']  = past_vals.std()  if len(past_vals) == window else np.nan

    pred = rf_model.predict(feat_row[X_train.columns])[0]
    preds_rf.append(pred)
    historico[data] = pred  # alimenta os próximos lags com a previsão

df_rf_2026 = pd.DataFrame({'previsao_rf': preds_rf}, index=future_index_rf)
df_rf_2026.index.name = 'data'
df_rf_2026.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
serie.iloc[-180:].plot(ax=ax, label='Histórico (6m)', color='steelblue')
df_rf_2026['previsao_rf'].plot(ax=ax, label='Previsão Random Forest 2026', color='seagreen')
ax.set_title('Previsão Random Forest — Volume Diário 2026')
ax.set_ylabel('Volume de Atendimentos')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
plt.tight_layout()
plt.show()

# Importância das features
feat_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
feat_imp.head(15).plot(kind='bar', title='Top 15 Features — Random Forest', color='seagreen')
plt.tight_layout()
plt.show()

print(f"\nTotal previsto 2026 (Random Forest): {df_rf_2026['previsao_rf'].sum():,.0f}")

## 4. Comparação dos Três Modelos

In [ ]:
df_comparacao = pd.DataFrame({
    'ARIMA':         df_arima['previsao_arima'],
    'Prophet':       df_prophet_2026['previsao_prophet'],
    'RandomForest':  df_rf_2026['previsao_rf'],
})
df_comparacao.index.name = 'data'

fig, ax = plt.subplots(figsize=(14, 6))
serie.iloc[-180:].plot(ax=ax, label='Histórico (6m)', color='steelblue', linewidth=1.5)
df_comparacao['ARIMA'].plot(ax=ax, label='ARIMA', color='tomato', linewidth=1.2)
df_comparacao['Prophet'].plot(ax=ax, label='Prophet', color='darkorange', linewidth=1.2)
df_comparacao['RandomForest'].plot(ax=ax, label='Random Forest', color='seagreen', linewidth=1.2)
ax.set_title('Comparação das Previsões — Volume Diário 2026')
ax.set_ylabel('Volume de Atendimentos')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
plt.tight_layout()
plt.show()

In [ ]:
# Resumo mensal
df_comparacao['mes'] = df_comparacao.index.to_period('M')
resumo_mensal = df_comparacao.groupby('mes')[['ARIMA', 'Prophet', 'RandomForest']].sum()
resumo_mensal.index = resumo_mensal.index.astype(str)

resumo_mensal.plot(kind='bar', figsize=(14, 5), title='Volume Mensal Previsto 2026 por Modelo')
plt.ylabel('Volume Total Mensal')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n--- Total Anual Previsto 2026 ---")
print(df_comparacao[['ARIMA', 'Prophet', 'RandomForest']].sum().apply(lambda x: f"{x:,.0f}"))

In [ ]:
# Exporta as previsões para CSV
df_export = df_comparacao.drop(columns=['mes'])
df_export.to_csv('../Bases/previsao_volume_2026.csv')
print("Arquivo salvo em: Bases/previsao_volume_2026.csv")
df_export.head()